In [4]:
import transformers
from transformers import AutoModelForSequenceClassification, DataCollatorWithPadding, PreTrainedTokenizerBase
from transformers import AutoTokenizer, TrainingArguments, default_data_collator

from trl import RewardConfig, RewardTrainer
from peft import LoraConfig 
import pandas as pd 
import torch
from datasets import load_dataset, DatasetDict
from huggingface_hub import login
import os
from typing import Dict

In [7]:
SEED = 42
SHUFFLE_SEED = 42
HF_DATASET_ID = "eZWALT/rlhf_reward_data_raw"  
HUB_REPO_ID = "eZWALT/rlhf_reward_splits_raw"  
PUSH_TO_HUB = False


SEED = 42
SHUFFLE_SEED = 42
ds = load_dataset(HF_DATASET_ID, split="train")   


# 2) shuffle then split to 80/10/10
# First shuffle the entire dataset (important to get a random split)
ds = ds.shuffle(seed=SHUFFLE_SEED)

# Split 80/20 (train / rest)
train_test = ds.train_test_split(test_size=0.20, seed=SEED)
train_ds = train_test["train"]          # ~80%
rest_ds = train_test["test"]            # ~20%

# Split the rest into half/half -> validation/test = 10% each
val_test = rest_ds.train_test_split(test_size=0.5, seed=SEED)
val_ds = val_test["train"]              # ~10%
test_ds = val_test["test"]              # ~10%

# Put into DatasetDict
dataset_dict = DatasetDict({
    "train": train_ds,
    "validation": val_ds,
    "test": test_ds
})

print(dataset_dict)
print("Train / Val / Test sizes:", len(dataset_dict["train"]), len(dataset_dict["validation"]), len(dataset_dict["test"]))

# 3a) Save locally for later use (optional)
dataset_dict.save_to_disk("../data/hf_rlhf_splits")

# 3b) Push the new split dataset to the Hub (optional)
if PUSH_TO_HUB:
    dataset_dict.push_to_hub(HUB_REPO_ID, private=True)
    print("Pushed split dataset to hub at:", HUB_REPO_ID)


DatasetDict({
    train: Dataset({
        features: ['prompt', 'chosen', 'rejected', 'model'],
        num_rows: 1200
    })
    validation: Dataset({
        features: ['prompt', 'chosen', 'rejected', 'model'],
        num_rows: 150
    })
    test: Dataset({
        features: ['prompt', 'chosen', 'rejected', 'model'],
        num_rows: 150
    })
})
Train / Val / Test sizes: 1200 150 150


Saving the dataset (1/1 shards): 100%|██████████| 150/150 [00:00<00:00, 18774.86 examples/s]


In [10]:
model = AutoModelForSequenceClassification.from_pretrained(
    "Qwen/Qwen2.5-0.5B-Instruct",
    dtype=torch.bfloat16,
    #num_labels=1 # If we ever train the reward model ever again `please use this, it should not have 2 heads that is nonsense
)

reward_config = RewardConfig(
    bf16=False,
    disable_dropout=False,
    remove_unused_columns=False,
    logging_steps=1,
    max_steps=1,
    #num_train_epochs=3,
    #data_seed=42,
)

processing_class = AutoTokenizer.from_pretrained("Qwen/Qwen2.5-0.5B-Instruct")


trainer = RewardTrainer(   
    model=model,
    train_dataset=train_ds,
    eval_dataset=rest_ds,
    args=reward_config,
    processing_class=processing_class,
)
trainer.train()

Some weights of Qwen2ForSequenceClassification were not initialized from the model checkpoint at Qwen/Qwen2.5-0.5B-Instruct and are newly initialized: ['score.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Filter: 100%|██████████| 300/300 [00:00<00:00, 5506.84 examples/s]
The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.
/home/walterjtv/.pyenv/versions/base/lib/python3.12/site-packages/torch/utils/data/dataloader.py:666: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)
You're using a Qwen2TokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a 

Step,Training Loss
1,1.539100


TrainOutput(global_step=1, training_loss=1.5390625, metrics={'train_runtime': 824.1161, 'train_samples_per_second': 0.01, 'train_steps_per_second': 0.001, 'total_flos': 0.0, 'train_loss': 1.5390625, 'epoch': 0.006666666666666667})

In [11]:
model

Qwen2ForSequenceClassification(
  (model): Qwen2Model(
    (embed_tokens): Embedding(151936, 896)
    (layers): ModuleList(
      (0-23): 24 x Qwen2DecoderLayer(
        (self_attn): Qwen2Attention(
          (q_proj): Linear(in_features=896, out_features=896, bias=True)
          (k_proj): Linear(in_features=896, out_features=128, bias=True)
          (v_proj): Linear(in_features=896, out_features=128, bias=True)
          (o_proj): Linear(in_features=896, out_features=896, bias=False)
        )
        (mlp): Qwen2MLP(
          (gate_proj): Linear(in_features=896, out_features=4864, bias=False)
          (up_proj): Linear(in_features=896, out_features=4864, bias=False)
          (down_proj): Linear(in_features=4864, out_features=896, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): Qwen2RMSNorm((896,), eps=1e-06)
        (post_attention_layernorm): Qwen2RMSNorm((896,), eps=1e-06)
      )
    )
    (norm): Qwen2RMSNorm((896,), eps=1e-06)
    (rota